In [1]:
from typing import TypedDict,Annotated
from langgraph.graph import StateGraph,START,END
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

In [2]:
load_dotenv()
llm = ChatOpenAI(
    model="qwen/qwen3.8-max:free",
    base_url="https://api.xkiro.com/v1"
)


In [4]:
sub_llm = ChatOpenAI(
    model="qwen/qwen3.8-max:free",
    base_url="https://api.xkiro.com/v1"
)

In [5]:
class SubState(TypedDict):
    input_text:str
    translated_text:str

In [17]:
def translate_text(state:SubState):
    prompt=f"""
    translate the following text to hindi
    keep it natural and clean and don't add extra content
    
    text:{state['input_text']}
    """.strip()
    translate_text=sub_llm.invoke(prompt).content

    return {"translated_text":translate_text}

In [18]:
subgraph_builder=StateGraph(SubState)
subgraph_builder.add_node("translate_text",translate_text)

subgraph_builder.add_edge(START,"translate_text")
subgraph_builder.add_edge("translate_text",END)

subgraph=subgraph_builder.compile()

In [19]:
class ParentState(TypedDict):
    question:str
    ans_eng:str
    ans_hin:str

In [20]:
def generate_ans(state:ParentState):
    answer=llm.invoke(f"you are a help full assistant. answer clearly.\n\nquestion:{state['question']}").content
    return {"ans_eng":answer}

In [21]:
def translate_ans(state:ParentState):
    result=subgraph.invoke({"input_text":state["ans_eng"]})
    return{"ans_hin":result["translated_text"]}

In [22]:
parent_builder=StateGraph(ParentState)

parent_builder.add_node("generate_ans",generate_ans)
parent_builder.add_node("translate_ans",translate_ans)

parent_builder.add_edge(START,"generate_ans")
parent_builder.add_edge("generate_ans","translate_ans")
parent_builder.add_edge("translate_ans",END)

In [23]:
parentGraph=parent_builder.compile()

In [24]:
parentGraph.invoke({"question":"what is machine learning?"})

{'question': 'what is machine learning?',
 'ans_eng': '**Machine learning** is a branch of artificial intelligence (AI) that focuses on building systems that can learn from data, identify patterns, and make decisions or predictions with minimal human intervention.\n\nInstead of being explicitly programmed with step-by-step rules for every possible scenario, a machine learning model is "trained" using large amounts of data. Through this training process, the system improves its performance over time.\n\nHere is a clear breakdown of how it works and its main types:\n\n### How It Works\n1. **Input Data:** You provide the system with data (e.g., thousands of pictures of cats and dogs).\n2. **Training:** The algorithm analyzes the data to find underlying patterns (e.g., ear shape, snout length).\n3. **Model Creation:** The system builds a mathematical model based on those patterns.\n4. **Prediction/Output:** When given new, unseen data (a new picture), the model uses what it learned to make